# Surface water fracture estimation

Now that we have properly adapted the data of previous satelites, we can start to develop a proper model for the estimation of the surface water fracture.

### Library loading

In [ ]:
# Import necessary libraries
import os
import xarray as xr
import pandas as pd
import time
import optuna
from tabulate import tabulate
from CIMR.SurfaceWaterFraction_ATBD_main.algorithm.processing.validation_data_processing import load_lut, unravel_freqpol, atmospheric_corrections

# Modelos
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
#from catboost import CatBoostRegressor

### Data loading

## Model selection

We'll start by selecting a few different models which we'll later compare in order to determine the most appropriate one.

In [ ]:
base_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=42)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.00001),
    "Bagging": BaggingRegressor(estimator=base_tree, n_estimators=20, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.00001, l1_ratio=0.5, random_state=42),
    "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=100, solver='adam', alpha=0.0001, random_state=42),
    "DecisionTree": DecisionTreeRegressor(max_depth=40, min_samples_split=20, min_samples_leaf=20 ,random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=20, max_depth=40, min_samples_split=20, min_samples_leaf=20),
    "XGBoost": XGBRegressor(n_estimators=20, learning_rate=0.1, max_depth=15, verbosity=0, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.5, max_depth=12),
    "AdaBoost": AdaBoostRegressor(estimator=base_tree, n_estimators=40, learning_rate=0.1, random_state=42)
}

## Model training and evaluation

Let's train and evaluate all the models so that we can get a general idea of their performance.

In [ ]:
def train_and_evaluate_models_list(X_train, y_train, X_test, y_test, models):
    results_all = {}

    for name, model in models.items() if isinstance(models, dict) else models:

        start_time = time.time()
        model.fit(X_train, y_train)
        elapsed_time = time.time() - start_time

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        results = {
            "MAE_train": mean_absolute_error(y_train, y_pred_train),
            "MSE_train": mean_squared_error(y_train, y_pred_train),
            "RMSE_train": root_mean_squared_error(y_train, y_pred_train),
            "MAPE_train": mean_absolute_percentage_error(y_train, y_pred_train),
            "R²_train": r2_score(y_train, y_pred_train),

            "MAE_test": mean_absolute_error(y_test, y_pred_test),
            "MSE_test": mean_squared_error(y_test, y_pred_test),
            "RMSE_test": root_mean_squared_error(y_test, y_pred_test),
            "MAPE_test": mean_absolute_percentage_error(y_test, y_pred_test),
            "R²_test": r2_score(y_test, y_pred_test),

            "Tiempo": round(elapsed_time, 4)
        }

        results_all[name] = results
    return results_all

def format_results_table(results_dict, tablefmt="github"):
    df = pd.DataFrame(results_dict).T  # filas=modelos, columnas=métricas
    df = df.round(6)

    return df

In [ ]:
results = train_and_evaluate_models_list(X_train, y_train, X_test, y_test, models)

df = format_results_table(results)

## Hiperparameter tuning

Now that we know which models work best, we can start tuning their hyperparameters to try and improve their performance.

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 25),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 10, log=True),
        "tree_method": "hist"
        }

    model = XGBRegressor(**params)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return -mean_squared_error(y_test, preds)

# Crear el estudio
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Mejores parámetros:", study.best_params)
print("Mejor score:", study.best_value)

## Explainability

The relation between SWF and the variables we are using to predict it is a physical one. For that reason, we consider that analizing the model in order to understand which variables are the most important in order to predict SWF is a good idea.